# **Start Section:**


In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Colab was not detected. Skipping Drive mount.')


In [ ]:
if 'google.colab' in sys.modules:
    !pip install --quiet numpy==1.26.4 pandas scipy scikit-learn matplotlib seaborn xlsxwriter openpyxl torch properscoring
else:
    print('Local runtime detected. Install dependencies manually if needed.')


In [ ]:
import os
if 'google.colab' in sys.modules:
    os.environ['PIP_CONSTRAINT'] = '/tmp/numpy_constraint.txt'
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt


# **Imports**


In [ ]:
import io
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import properscoring as ps
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# Go to find and replace button and replace (Data_folder) with your folder name.
# Rename your train and test dataset as train.csv and test.csv.
target_column = None
random_seed = 42
quick_run = False


In [ ]:
feature_names = ['Qt', 'Qt-1', 'St-1']


In [ ]:
train_data_path = './drive/MyDrive/Data_folder/Data/train.csv'
test_data_path = './drive/MyDrive/Data_folder/Data/test.csv'
output_folder = './drive/MyDrive/Data_folder/Conformal_Predictions(AdaptiveCoveragePolicies)'


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_demo_dataframe(n_rows, feature_names, seed=42, is_train=True):
    local_feature_names = feature_names if len(feature_names) >= 3 else ['X1', 'X2', 'X3']
    rng = np.random.default_rng(seed + (0 if is_train else 99))
    data = {name: rng.normal(loc=0.0, scale=1.0, size=n_rows) for name in local_feature_names}
    frame = pd.DataFrame(data)
    nonlinear_term = 0.8 * np.sin(frame[local_feature_names[0]].values)
    interaction_term = 0.5 * frame[local_feature_names[1]].values * frame[local_feature_names[2]].values
    trend_term = 0.3 * frame[local_feature_names[0]].values ** 2
    noise = rng.normal(loc=0.0, scale=0.35 if is_train else 0.4, size=n_rows)
    frame['Target'] = 12.0 + 3.2 * frame[local_feature_names[0]].values - 1.7 * frame[local_feature_names[1]].values + 2.1 * frame[local_feature_names[2]].values + nonlinear_term + interaction_term - trend_term + noise
    return frame


def read_csv_from_candidates(path_candidates):
    for path in path_candidates:
        path = Path(path)
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if not df.empty:
                print(f'Loaded data from: {path}')
                return df
        except Exception:
            continue
    return pd.DataFrame()


set_seed(random_seed)
train_demo = build_demo_dataframe(280, feature_names, seed=random_seed, is_train=True)
test_demo = build_demo_dataframe(100, feature_names, seed=random_seed, is_train=False)

train_candidates = [
    train_data_path,
    '/content/drive/MyDrive/Data_folder/Data/train.csv',
    'Data_folder/Data/train.csv',
]
test_candidates = [
    test_data_path,
    '/content/drive/MyDrive/Data_folder/Data/test.csv',
    'Data_folder/Data/test.csv',
]

train_data = read_csv_from_candidates(train_candidates)
test_data = read_csv_from_candidates(test_candidates)

if train_data.empty:
    print('Warning: training CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.')
    train_data = train_demo.copy()
if test_data.empty:
    print('Warning: testing CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.')
    test_data = test_demo.copy()


In [ ]:
print('Shape of training data:', train_data.shape)
print('First 5 rows of training data:', train_data.head(5))
print('Shape of test data:', test_data.shape)
print('First 5 rows of test data:', test_data.head(5))


In [ ]:
if target_column is None:
    target_column = train_data.columns[-1]

available_feature_names = [name for name in feature_names if name in train_data.columns]
if len(available_feature_names) == len(feature_names):
    selected_feature_names = feature_names
else:
    selected_feature_names = train_data.columns[:-1].tolist()
    print(f'Feature names were adjusted automatically to match the dataset columns: {selected_feature_names}')

X_train_full = train_data[selected_feature_names].copy()
y_train_full = train_data[target_column].copy()
X_test = test_data[selected_feature_names].copy()
y_test = test_data[target_column].copy()

X_train_base_raw, X_calib_raw, y_train_base_raw, y_calib_raw = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.35,
    random_state=random_seed,
)

scaler = StandardScaler()
X_train_base = scaler.fit_transform(X_train_base_raw.to_numpy(dtype=np.float32))
X_calib = scaler.transform(X_calib_raw.to_numpy(dtype=np.float32))
X_test_scaled = scaler.transform(X_test.to_numpy(dtype=np.float32))

y_train_base = y_train_base_raw.to_numpy(dtype=np.float32).reshape(-1)
y_calib = y_calib_raw.to_numpy(dtype=np.float32).reshape(-1)
y_test_array = y_test.to_numpy(dtype=np.float32).reshape(-1)

print('Selected feature names:', selected_feature_names)
print('Target column:', target_column)
print('Base training split:', X_train_base.shape, y_train_base.shape)
print('Calibration split:', X_calib.shape, y_calib.shape)
print('Test split:', X_test_scaled.shape, y_test_array.shape)


# **Functions:**


In [ ]:
class AlphaNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.initialize_to_one()

    def initialize_to_one(self):
        with torch.no_grad():
            self.fc2.weight.data.normal_(0, 0.01)
            self.fc2.bias.data.fill_(5.0)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.sigmoid(self.fc2(x)).squeeze(-1)


def compute_scores_without_index(base_model, X_calib, y_calib, holdout_idx):
    loo_X = np.delete(X_calib, holdout_idx, axis=0)
    loo_y = np.delete(y_calib, holdout_idx, axis=0)
    return np.abs(base_model.predict(loo_X) - loo_y)


def build_loo_feature_matrix(base_model, X_calib, y_calib):
    train_inputs = []
    for idx in range(len(X_calib)):
        scores = compute_scores_without_index(base_model, X_calib, y_calib, idx)
        train_inputs.append(torch.tensor([scores.sum()], dtype=torch.float32))
    return torch.stack(train_inputs)


def interval_size(scores, alpha, eps=1e-6):
    n = len(scores)
    denominator = torch.clamp(alpha * (n + 1) - 1.0, min=eps)
    return 2.0 * scores.sum() / denominator


def train_alpha_net(
    lambda_reg,
    base_model,
    X_calib,
    y_calib,
    X_train_alpha,
    run_id=0,
    num_epochs=50,
    batch_size=32,
    learning_rate=1e-3,
    print_every=10,
    alpha_clip=1e-3,
    eps=1e-6,
    seed=42,
):
    set_seed(seed + run_id)
    train_dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(device)
    optimizer = optim.Adam(alpha_net.parameters(), lr=learning_rate)

    all_losses, all_sizes, all_alphas = [], [], []

    for epoch in range(num_epochs):
        alpha_net.train()
        epoch_losses, epoch_sizes, epoch_alphas = [], [], []

        for x_batch, idx_batch in train_loader:
            x_batch = x_batch.to(device)
            alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)

            batch_sizes = []
            for j, idx in enumerate(idx_batch.tolist()):
                scores_np = compute_scores_without_index(base_model, X_calib, y_calib, idx)
                scores = torch.tensor(scores_np, dtype=torch.float32, device=device)
                size_j = interval_size(scores, alpha_pred[j], eps=eps)
                batch_sizes.append(size_j)

            batch_sizes = torch.stack(batch_sizes)
            loss = (batch_sizes + lambda_reg * alpha_pred).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(float(loss.item()))
            epoch_sizes.append(float(batch_sizes.mean().item()))
            epoch_alphas.append(float(alpha_pred.mean().item()))

        all_losses.append(float(np.mean(epoch_losses)))
        all_sizes.append(float(np.mean(epoch_sizes)))
        all_alphas.append(float(np.mean(epoch_alphas)))

        if (epoch % print_every == 0) or (epoch == num_epochs - 1):
            print(
                f'Lambda {lambda_reg} | Run {run_id + 1} | Epoch {epoch + 1}/{num_epochs} | '
                f'Loss: {all_losses[-1]:.5f} | Mean Size: {all_sizes[-1]:.5f} | Mean Alpha: {all_alphas[-1]:.5f}'
            )

    alpha_net.eval()
    return np.array(all_losses), np.array(all_sizes), np.array(all_alphas), alpha_net


def normalize_confidence(confidence_values):
    confidence_values = np.asarray(confidence_values, dtype=float)
    cmin, cmax = confidence_values.min(), confidence_values.max()
    if np.isclose(cmin, cmax):
        return np.ones_like(confidence_values)
    return (confidence_values - cmin) / (cmax - cmin)


@torch.no_grad()
def evaluate_calibration_loo(alpha_net, base_model, X_calib, y_calib, alpha_clip=1e-3, eps=1e-6):
    rows = []
    for idx in range(len(X_calib)):
        scores_np = compute_scores_without_index(base_model, X_calib, y_calib, idx)
        feature_tensor = torch.tensor([[scores_np.sum()]], dtype=torch.float32, device=device)
        alpha_value = float(torch.clamp(alpha_net(feature_tensor), min=alpha_clip, max=1.0 - alpha_clip).item())

        scores_tensor = torch.tensor(scores_np, dtype=torch.float32, device=device)
        interval_width = float(interval_size(scores_tensor, torch.tensor(alpha_value, device=device), eps=eps).item())

        pred_i = float(base_model.predict(X_calib[idx].reshape(1, -1))[0])
        actual_i = float(y_calib[idx])

        lower_95 = pred_i - 0.5 * interval_width
        upper_95 = pred_i + 0.5 * interval_width
        covered_95 = int((actual_i >= lower_95) and (actual_i <= upper_95))

        rows.append(
            {
                'Index': idx,
                'Pred': pred_i,
                'Actual': actual_i,
                'Alpha': alpha_value,
                'Interval_Width': interval_width,
                'Lower_95': lower_95,
                'Upper_95': upper_95,
                'Covered_95': covered_95,
            }
        )

    return pd.DataFrame(rows)


@torch.no_grad()
def predict_test_with_alpha_net(alpha_net, base_model, X_calib, y_calib, X_test_scaled, y_test_array, alpha_clip=1e-3, eps=1e-6):
    scores_calib = np.abs(base_model.predict(X_calib) - y_calib)
    feat = torch.tensor([[scores_calib.sum()]], dtype=torch.float32, device=device)

    alpha_value = float(torch.clamp(alpha_net(feat), min=alpha_clip, max=1.0 - alpha_clip).item())
    scores_tensor = torch.tensor(scores_calib, dtype=torch.float32, device=device)
    interval_width = float(interval_size(scores_tensor, torch.tensor(alpha_value, device=device), eps=eps).item())

    pred_mean = base_model.predict(X_test_scaled).reshape(-1)
    y_true = np.asarray(y_test_array).reshape(-1)

    sigma_hat = np.full_like(pred_mean, fill_value=interval_width / 2.0, dtype=float)
    confidence = np.full_like(pred_mean, fill_value=max(0.0, 1.0 - alpha_value), dtype=float)
    confidence_normalized = normalize_confidence(confidence)
    absolute_error = np.abs(y_true - pred_mean)

    predictions_df = pd.DataFrame(
        {
            'Mean': pred_mean,
            'Sigma_Hat_Raw': sigma_hat,
            'Sigma_Hat': sigma_hat,
            'StdDev': sigma_hat,
            'Confidence': confidence,
            'Confidence_Normalized': confidence_normalized,
            'RawUncertainty': sigma_hat,
            'Alpha': alpha_value,
            'Absolute_Error': absolute_error,
            'Actual': y_true,
        }
    )

    for k in [1, 2, 3]:
        predictions_df[f'Lower_{k}Sigma'] = predictions_df['Mean'] - k * predictions_df['Sigma_Hat']
        predictions_df[f'Upper_{k}Sigma'] = predictions_df['Mean'] + k * predictions_df['Sigma_Hat']
        predictions_df[f'Coverage_{k}Sigma'] = (
            (predictions_df['Actual'] >= predictions_df[f'Lower_{k}Sigma'])
            & (predictions_df['Actual'] <= predictions_df[f'Upper_{k}Sigma'])
        ).astype(int)

    z_95 = 1.96
    predictions_df['Lower_95'] = predictions_df['Mean'] - z_95 * predictions_df['Sigma_Hat']
    predictions_df['Upper_95'] = predictions_df['Mean'] + z_95 * predictions_df['Sigma_Hat']
    predictions_df['Coverage_95'] = (
        (predictions_df['Actual'] >= predictions_df['Lower_95'])
        & (predictions_df['Actual'] <= predictions_df['Upper_95'])
    ).astype(int)

    return predictions_df, alpha_value, interval_width


In [ ]:
def moving_average(arr, window_size):
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return arr
    window_size = max(1, min(window_size, len(arr)))
    if window_size == 1:
        return arr
    kernel = np.ones(window_size, dtype=float) / float(window_size)
    return np.convolve(arr, kernel, mode='valid')


def smooth_all(arr_2d, window):
    smoothed_runs = []
    for run in np.asarray(arr_2d):
        smoothed_runs.append(moving_average(run, window))
    smoothed_runs = np.asarray(smoothed_runs)
    return np.nanmean(smoothed_runs, axis=0), np.nanstd(smoothed_runs, axis=0)


def compute_regression_ece(sigma_values, absolute_errors, n_bins=10):
    sigma_values = np.asarray(sigma_values, dtype=float)
    absolute_errors = np.asarray(absolute_errors, dtype=float)
    quantile_edges = np.quantile(sigma_values, np.linspace(0, 1, n_bins + 1))
    quantile_edges[0] -= 1e-8
    quantile_edges[-1] += 1e-8

    total = len(sigma_values)
    ece = 0.0
    rows = []
    for idx in range(n_bins):
        lower = quantile_edges[idx]
        upper = quantile_edges[idx + 1]
        mask = (sigma_values >= lower) & (sigma_values < upper)
        if not np.any(mask):
            continue
        mean_sigma = sigma_values[mask].mean()
        mean_error = absolute_errors[mask].mean()
        weight = mask.sum() / total
        ece += weight * abs(mean_sigma - mean_error)
        rows.append(
            {
                'Bin': idx + 1,
                'Bin_Size': int(mask.sum()),
                'Mean_Sigma': mean_sigma,
                'Mean_Absolute_Error': mean_error,
                'Absolute_Gap': abs(mean_sigma - mean_error),
            }
        )

    return float(ece), pd.DataFrame(rows)


def compute_paper_metrics(predictions_df, model_name='ACP'):
    absolute_error = predictions_df['Absolute_Error'].values
    sigma = np.clip(predictions_df['Sigma_Hat'].values, 1e-8, None)

    coverage_1 = np.mean(absolute_error <= sigma)
    coverage_2 = np.mean(absolute_error <= 2.0 * sigma)
    coverage_3 = np.mean(absolute_error <= 3.0 * sigma)
    coverage_95 = predictions_df['Coverage_95'].mean()
    average_95_width = (predictions_df['Upper_95'] - predictions_df['Lower_95']).mean()
    ece_reg, ece_bins_df = compute_regression_ece(sigma, absolute_error, n_bins=10)

    pearson_value = pearsonr(sigma, absolute_error)[0] if len(sigma) > 1 else np.nan
    spearman_value = spearmanr(sigma, absolute_error)[0] if len(sigma) > 1 else np.nan
    rmse = np.sqrt(mean_squared_error(predictions_df['Actual'], predictions_df['Mean']))
    mae = mean_absolute_error(predictions_df['Actual'], predictions_df['Mean'])

    summary_df = pd.DataFrame(
        [
            {
                'Model': model_name,
                'Cov@1Sigma': coverage_1,
                'Cov@2Sigma': coverage_2,
                'Cov@3Sigma': coverage_3,
                'Coverage_95Interval': coverage_95,
                'Average_95Interval_Width': average_95_width,
                'Pearson': pearson_value,
                'Spearman': spearman_value,
                'ECE_reg': ece_reg,
                'RMSE': rmse,
                'MAE': mae,
                'Global_Alpha': float(predictions_df['Alpha'].iloc[0]),
            }
        ]
    )

    return summary_df, ece_bins_df


def insert_figure_into_worksheet(writer, sheet_name, figure, image_cell='H2'):
    image_buffer = io.BytesIO()
    figure.savefig(image_buffer, format='png', dpi=200, bbox_inches='tight')
    image_buffer.seek(0)
    worksheet = writer.sheets[sheet_name]
    worksheet.insert_image(image_cell, 'plot.png', {'image_data': image_buffer})
    plt.close(figure)


def generate_and_save_plots(predictions_df, calibration_loo_df, excel_path, all_results, smoothing_window=8):
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

        fig0, axs = plt.subplots(1, 3, figsize=(16, 4.5))
        colors = ['#0077bb', '#cc3311', '#44aa99', '#ee7733', '#009988']
        sorted_lambdas = sorted(all_results.keys())
        training_summary_rows = []

        for lam_idx, lam in enumerate(sorted_lambdas):
            color = colors[lam_idx % len(colors)]
            losses = all_results[lam]['losses']
            sizes = all_results[lam]['sizes']
            alphas = all_results[lam]['alphas']

            mean_loss, std_loss = smooth_all(losses, smoothing_window)
            mean_size, std_size = smooth_all(sizes, smoothing_window)
            mean_alpha, std_alpha = smooth_all(alphas, smoothing_window)

            epochs = np.arange(1, len(mean_loss) + 1)
            axs[0].plot(epochs, mean_loss, label=f'lambda={lam}', color=color)
            axs[0].fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, color=color, alpha=0.25)

            axs[1].plot(epochs, mean_size, label=f'lambda={lam}', color=color)
            axs[1].fill_between(epochs, mean_size - std_size, mean_size + std_size, color=color, alpha=0.25)

            axs[2].plot(epochs, mean_alpha, label=f'lambda={lam}', color=color)
            axs[2].fill_between(epochs, mean_alpha - std_alpha, mean_alpha + std_alpha, color=color, alpha=0.25)

            training_summary_rows.append(
                {
                    'Lambda': lam,
                    'Final_Loss_Mean': float(np.mean(losses[:, -1])),
                    'Final_Size_Mean': float(np.mean(sizes[:, -1])),
                    'Final_Alpha_Mean': float(np.mean(alphas[:, -1])),
                }
            )

        axs[0].set_title('Training Loss')
        axs[0].set_xlabel('Epoch')
        axs[0].set_ylabel('Loss')
        axs[0].grid(True)

        axs[1].set_title('Mean Interval Size')
        axs[1].set_xlabel('Epoch')
        axs[1].set_ylabel('Interval Width')
        axs[1].grid(True)

        axs[2].set_title('Mean Alpha')
        axs[2].set_xlabel('Epoch')
        axs[2].set_ylabel('Alpha')
        axs[2].grid(True)
        axs[2].legend(frameon=True)

        training_summary_df = pd.DataFrame(training_summary_rows)
        training_summary_df.to_excel(writer, sheet_name='Training_Curves', index=False)
        insert_figure_into_worksheet(writer, 'Training_Curves', fig0, image_cell='H2')

        ordered_df = predictions_df.sort_values('Actual').reset_index(drop=True)
        ordered_df.to_excel(writer, sheet_name='Bands_95Coverage', index=False)

        fig1, ax1 = plt.subplots(figsize=(12, 8))
        ax1.plot(ordered_df.index, ordered_df['Mean'], color='black', label='Prediction', linewidth=1.2)
        ax1.scatter(ordered_df.index, ordered_df['Actual'], color='darkorange', s=18, alpha=0.8, label='Actual')
        ax1.fill_between(
            ordered_df.index,
            ordered_df['Lower_95'],
            ordered_df['Upper_95'],
            color='skyblue',
            alpha=0.25,
            label='95% interval',
        )
        ax1.set_title('ACP Prediction with 95% Interval')
        ax1.set_xlabel('Ordered Test Sample')
        ax1.set_ylabel('Target Value')
        ax1.legend(loc='best')
        insert_figure_into_worksheet(writer, 'Bands_95Coverage', fig1)

        calibration_loo_df.to_excel(writer, sheet_name='Calibration_LOO', index=False)
        fig2, ax2 = plt.subplots(figsize=(10, 7))
        sns.histplot(calibration_loo_df['Interval_Width'], bins=15, kde=True, ax=ax2)
        ax2.set_title('Calibration LOO Interval Width Distribution')
        ax2.set_xlabel('Interval Width')
        ax2.set_ylabel('Frequency')
        insert_figure_into_worksheet(writer, 'Calibration_LOO', fig2, image_cell='J2')

        fig3, ax3 = plt.subplots(figsize=(10, 7))
        sns.scatterplot(x='Sigma_Hat', y='Absolute_Error', data=predictions_df, ax=ax3)
        ax3.set_title('ACP Uncertainty vs Absolute Error')
        ax3.set_xlabel('Interval Half Width')
        ax3.set_ylabel('Absolute Error')
        pd.DataFrame({'Sigma_Hat': predictions_df['Sigma_Hat'], 'Absolute_Error': predictions_df['Absolute_Error']}).to_excel(
            writer, sheet_name='Uncertainty_vs_Error', index=False
        )
        insert_figure_into_worksheet(writer, 'Uncertainty_vs_Error', fig3)

        fig4, ax4 = plt.subplots(figsize=(10, 7))
        sns.scatterplot(x='Actual', y='Mean', data=predictions_df, ax=ax4)
        diagonal_min = min(predictions_df['Actual'].min(), predictions_df['Mean'].min())
        diagonal_max = max(predictions_df['Actual'].max(), predictions_df['Mean'].max())
        ax4.plot([diagonal_min, diagonal_max], [diagonal_min, diagonal_max], linestyle='--', color='red')
        ax4.set_title('Predicted Target vs Actual Target')
        ax4.set_xlabel('Actual Target')
        ax4.set_ylabel('Predicted Mean Target')
        pd.DataFrame({'Actual': predictions_df['Actual'], 'Mean': predictions_df['Mean']}).to_excel(
            writer, sheet_name='Predicted_vs_Actual', index=False
        )
        insert_figure_into_worksheet(writer, 'Predicted_vs_Actual', fig4)


def generate_and_save_paper_metrics(predictions_df, excel_path, model_name='ACP'):
    paper_metrics_df, ece_bins_df = compute_paper_metrics(predictions_df, model_name=model_name)

    coverage_df = pd.DataFrame(
        [
            {'Scale': '1Sigma', 'Coverage': predictions_df['Coverage_1Sigma'].mean(), 'Target': 0.68},
            {'Scale': '95PercentInterval', 'Coverage': predictions_df['Coverage_95'].mean(), 'Target': 0.95},
            {'Scale': '2Sigma', 'Coverage': predictions_df['Coverage_2Sigma'].mean(), 'Target': 0.95},
            {'Scale': '3Sigma', 'Coverage': predictions_df['Coverage_3Sigma'].mean(), 'Target': 0.997},
        ]
    )

    confidence_bins = pd.qcut(predictions_df['Confidence_Normalized'], q=min(10, len(predictions_df)), duplicates='drop')
    confidence_bin_df = (
        predictions_df.assign(Confidence_Bin=confidence_bins)
        .groupby('Confidence_Bin', observed=False)
        .agg(
            Mean_Confidence=('Confidence_Normalized', 'mean'),
            Mean_Absolute_Error=('Absolute_Error', 'mean'),
            Mean_Sigma=('Sigma_Hat', 'mean'),
            Count=('Confidence_Normalized', 'size'),
        )
        .reset_index()
    )

    percentile_df = pd.DataFrame(
        {
            'Percentile': [50, 75, 90, 95, 99],
            'Sigma_Hat_Threshold': np.percentile(predictions_df['Sigma_Hat'], [50, 75, 90, 95, 99]),
            'RawUncertainty_Threshold': np.percentile(predictions_df['RawUncertainty'], [50, 75, 90, 95, 99]),
        }
    )

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        paper_metrics_df.to_excel(writer, sheet_name='Paper_Metrics', index=False)
        coverage_df.to_excel(writer, sheet_name='Coverage_Details', index=False)
        ece_bins_df.to_excel(writer, sheet_name='ECE_Bins', index=False)
        confidence_bin_df.to_excel(writer, sheet_name='Confidence_Bins', index=False)
        percentile_df.to_excel(writer, sheet_name='Uncertainty_Thresholds', index=False)


def generate_and_save_secondary_metrics(predictions_df, excel_path, model_name='ACP'):
    mu_pred = predictions_df['Mean'].values
    sigma_pred = np.clip(predictions_df['Sigma_Hat'].values, 1e-8, None)
    y_true = predictions_df['Actual'].values

    crps_values = ps.crps_gaussian(y_true, mu=mu_pred, sig=sigma_pred)
    nll_values = -0.5 * np.log(2 * np.pi * sigma_pred**2) - ((y_true - mu_pred) ** 2) / (2 * sigma_pred**2)

    secondary_df = pd.DataFrame({'Mean': mu_pred, 'Sigma_Hat': sigma_pred, 'CRPS': crps_values, 'LogLikelihood': nll_values})

    summary_df = pd.DataFrame(
        [
            {
                'Model': model_name,
                'Mean_CRPS': float(np.mean(crps_values)),
                'Mean_Negative_LogLikelihood': float(np.mean(-nll_values)),
                'Coverage_95Interval': float(predictions_df['Coverage_95'].mean()),
            }
        ]
    )

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        secondary_df.to_excel(writer, sheet_name='Secondary_Compatibility', index=False)
        summary_df.to_excel(writer, sheet_name='Secondary_Summary', index=False)


def build_matrix_evaluation(predictions_df, model_name='ACP'):
    paper_metrics_df, _ = compute_paper_metrics(predictions_df, model_name=model_name)
    matrix_df = paper_metrics_df.copy()
    matrix_df['Mean_Sigma_Hat'] = predictions_df['Sigma_Hat'].mean()
    matrix_df['Mean_RawUncertainty'] = predictions_df['RawUncertainty'].mean()
    matrix_df['Coverage_95Interval'] = predictions_df['Coverage_95'].mean()
    matrix_df['Average_95Interval_Width'] = (predictions_df['Upper_95'] - predictions_df['Lower_95']).mean()
    return matrix_df


In [ ]:
folder_path = '/content/drive/MyDrive/Data_folder'
if 'google.colab' in sys.modules:
    os.makedirs(folder_path, exist_ok=True)

if 'google.colab' in sys.modules:
    output_candidates = [
        Path(output_folder),
        Path('/content/drive/MyDrive/Data_folder/Conformal_Predictions(AdaptiveCoveragePolicies)'),
    ]
else:
    output_candidates = [
        Path('Data_folder/Conformal_Predictions(AdaptiveCoveragePolicies)'),
        Path(output_folder),
    ]

for candidate in output_candidates:
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        resolved_output_folder = candidate
        break
    except Exception:
        continue
else:
    raise RuntimeError('Could not create an output folder for ACP results.')


# **Adaptive Coverage Policies (ACP)**


In [ ]:
training_config = {
    'seed': random_seed,
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'print_every': 10,
    'alpha_clip': 1e-3,
    'eps': 1e-6,
    'lambdas': [10, 20, 50],
    'num_runs': 4,
    'smoothing_window': 8,
}

if quick_run:
    training_config.update({'epochs': 12, 'num_runs': 2, 'print_every': 4, 'smoothing_window': 3})

training_config


In [ ]:
set_seed(training_config['seed'])

base_model = LinearRegression()
base_model.fit(X_train_base, y_train_base)

X_train_alpha = build_loo_feature_matrix(base_model, X_calib, y_calib)

all_results = {}
trained_models = {}
history_rows = []

for lam in training_config['lambdas']:
    print(); print(f'Starting training for lambda={lam}')
    losses_runs, sizes_runs, alphas_runs = [], [], []
    alpha_models_for_lambda = []

    for run in range(training_config['num_runs']):
        losses, sizes, alphas, alpha_model = train_alpha_net(
            lambda_reg=lam,
            base_model=base_model,
            X_calib=X_calib,
            y_calib=y_calib,
            X_train_alpha=X_train_alpha,
            run_id=run,
            num_epochs=training_config['epochs'],
            batch_size=training_config['batch_size'],
            learning_rate=training_config['learning_rate'],
            print_every=training_config['print_every'],
            alpha_clip=training_config['alpha_clip'],
            eps=training_config['eps'],
            seed=training_config['seed'],
        )

        losses_runs.append(losses)
        sizes_runs.append(sizes)
        alphas_runs.append(alphas)
        alpha_models_for_lambda.append(alpha_model)

        for epoch_idx in range(len(losses)):
            history_rows.append(
                {
                    'Lambda': lam,
                    'Run': run + 1,
                    'Epoch': epoch_idx + 1,
                    'Loss': float(losses[epoch_idx]),
                    'Mean_Size': float(sizes[epoch_idx]),
                    'Mean_Alpha': float(alphas[epoch_idx]),
                }
            )

    all_results[lam] = {'losses': np.array(losses_runs), 'sizes': np.array(sizes_runs), 'alphas': np.array(alphas_runs)}
    trained_models[lam] = alpha_models_for_lambda

training_history_df = pd.DataFrame(history_rows)

selection_rows = []
for lam, model_list in trained_models.items():
    for run_idx, alpha_model in enumerate(model_list):
        calib_df = evaluate_calibration_loo(
            alpha_model,
            base_model,
            X_calib,
            y_calib,
            alpha_clip=training_config['alpha_clip'],
            eps=training_config['eps'],
        )

        selection_rows.append(
            {
                'Lambda': lam,
                'Run': run_idx + 1,
                'Calibration_Mean_Width': float(calib_df['Interval_Width'].mean()),
                'Calibration_Coverage_95': float(calib_df['Covered_95'].mean()),
            }
        )

model_selection_df = pd.DataFrame(selection_rows)
model_selection_df['Selection_Score'] = model_selection_df['Calibration_Mean_Width'] + 25.0 * np.abs(
    model_selection_df['Calibration_Coverage_95'] - 0.95
)

best_row = model_selection_df.sort_values('Selection_Score').iloc[0]
best_lambda = int(best_row['Lambda'])
best_run_index = int(best_row['Run']) - 1

alpha_net = trained_models[best_lambda][best_run_index]
calibration_loo_df = evaluate_calibration_loo(
    alpha_net,
    base_model,
    X_calib,
    y_calib,
    alpha_clip=training_config['alpha_clip'],
    eps=training_config['eps'],
)

print(f'Selected model -> lambda: {best_lambda}, run: {best_run_index + 1}')
print(model_selection_df.sort_values('Selection_Score').head())


In [ ]:
predictions_ACP_df, global_alpha, global_interval_width = predict_test_with_alpha_net(
    alpha_net,
    base_model,
    X_calib,
    y_calib,
    X_test_scaled,
    y_test_array,
    alpha_clip=training_config['alpha_clip'],
    eps=training_config['eps'],
)

print(f'Global alpha used on test: {global_alpha:.6f}')
print(f'Global interval width used on test: {global_interval_width:.6f}')
print(predictions_ACP_df.head())


In [ ]:
acp_excel_path = resolved_output_folder / 'Adaptive_Coverage_Policies(ACP).xlsx'
generate_and_save_plots(
    predictions_ACP_df,
    calibration_loo_df,
    acp_excel_path,
    all_results,
    smoothing_window=training_config['smoothing_window'],
)
print(f'Saved detailed ACP outputs to: {acp_excel_path}')


In [ ]:
generate_and_save_paper_metrics(predictions_ACP_df, acp_excel_path, model_name='ACP')
generate_and_save_secondary_metrics(predictions_ACP_df, acp_excel_path, model_name='ACP')

with pd.ExcelWriter(acp_excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    training_history_df.to_excel(writer, sheet_name='Training_History', index=False)
    model_selection_df.to_excel(writer, sheet_name='Model_Selection', index=False)
    calibration_loo_df.to_excel(writer, sheet_name='Calibration_LOO_Table', index=False)

print('Saved ACP paper metrics, secondary metrics, and history sheets.')


# **Matrix Evaulation**


In [ ]:
matrix_evaluation_df = build_matrix_evaluation(predictions_ACP_df, model_name='ACP')
matrix_excel_path = resolved_output_folder / 'Matrix Evaluation.xlsx'

with pd.ExcelWriter(matrix_excel_path, engine='xlsxwriter') as writer:
    matrix_evaluation_df.to_excel(writer, sheet_name='Matrix Evaluation', index=False)

print(matrix_evaluation_df)
print(f'Saved matrix evaluation to: {matrix_excel_path}')
